In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Setup Time-Series Data
metrics_data = [
    ("User_A", "2026-06-01", 120),
    ("User_A", "2026-06-02", 135),
    ("User_A", "2026-06-03", 150),
    ("User_A", "2026-06-04", 110),
    ("User_B", "2026-06-01", 200),
    ("User_B", "2026-06-02", 210),
]
columns = ["UserID", "Date", "Latency_ms"]

df_metrics = spark.createDataFrame(metrics_data, columns)

In [0]:
running_window = (
    Window.partitionBy("UserID")
    .orderBy("Date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_window_res = df_metrics.withColumn("Running_Total_Latency", F.sum("Latency_ms").over(running_window))

display(df_window_res)

UserID,Date,Latency_ms,Running_Total_Latency
User_A,2026-06-01,120,120
User_A,2026-06-02,135,255
User_A,2026-06-03,150,405
User_A,2026-06-04,110,515
User_B,2026-06-01,200,200
User_B,2026-06-02,210,410


Your Hands-On Mission
I want you to build a chained PySpark pipeline that flattens this nested data and cleans it up.

Write a query that does the following:

- Start with df_decants.

- Use .withColumn() to create a new column named SingleNote. The value of this column should be the exploded version of NotesArray. (Hint: F.explode(F.col("NotesArray"))).

- We need to do some quality control. Filter the DataFrame to remove any rows where the SingleNote is "Lavender" (since aromatic/lavender-heavy profiles can completely ruin a good fresh-spicy batch).

- Drop the original NotesArray column since we don't need the nested list anymore. (Hint: .drop()).

- Save the output to a variable and display() it.

In [0]:
import pyspark.sql.functions as F

# Setup: Decant Inventory with Arrays
decant_data = [
    ("CDN Iconic", "Citrus", ["Grapefruit", "Mint", "Nutmeg", "Ginger"]),
    ("Modest Une", "Fresh Spicy", ["Pepper", "Lavender", "Citrus", "Vetiver"]),
    ("Terre d'Hermes", "Woody", ["Orange", "Grapefruit", "Flint", "Oakmoss"])
]
columns = ["Fragrance", "Profile", "NotesArray"]

df_decants = spark.createDataFrame(decant_data, columns)
display(df_decants)

Fragrance,Profile,NotesArray
CDN Iconic,Citrus,"List(Grapefruit, Mint, Nutmeg, Ginger)"
Modest Une,Fresh Spicy,"List(Pepper, Lavender, Citrus, Vetiver)"
Terre d'Hermes,Woody,"List(Orange, Grapefruit, Flint, Oakmoss)"


In [0]:
df_single_note = (df_decants
                    .withColumn("SingleNote", F.explode(F.col("NotesArray")))
                    .filter(F.col("SingleNote")!="Lavender")
                    .drop("NotesArray")
                )
df_single_note.display()      

Fragrance,Profile,SingleNote
CDN Iconic,Citrus,Grapefruit
CDN Iconic,Citrus,Mint
CDN Iconic,Citrus,Nutmeg
CDN Iconic,Citrus,Ginger
Modest Une,Fresh Spicy,Pepper
Modest Une,Fresh Spicy,Citrus
Modest Une,Fresh Spicy,Vetiver
Terre d'Hermes,Woody,Orange
Terre d'Hermes,Woody,Grapefruit
Terre d'Hermes,Woody,Flint


# The Crucible Challenge #1: IoT Telemetry & Service Logs

Now, write a fully chained PySpark pipeline (or split it into logical DataFrames if it is cleaner) that does the following:

- Read the JSON file from your volume into a DataFrame.

- Read the CSV file from your volume into a separate DataFrame. (Make sure the CSV reader picks up the header row! You will need to pass an option for this.)

- Filter the telemetry data to only include the "CB300R-SpartanRed" model.

- Extract the tilt_angle from the nested telemetry struct into its own top-level column called Incident_Tilt. (Hint: Use dot notation).

- Filter out normal riding data by only keeping rows where the Incident_Tilt is greater than 60 degrees.

- Explode the warnings array so each warning gets its own row.

- Join this filtered, flattened JSON data with the CSV maintenance data.

- Select and display only these final columns: date, Incident_Tilt, warnings (the exploded version), service_action, and cost_inr.

In [0]:
import pyspark.sql.functions as F

df_telemetry_json = spark.read.json("/Volumes/dbacademy/default/tutorials/raw_telemetry.json")

df_maintenance_logs = (spark.read
                       .option("header", True)
                       .option("inferSchema", True)
                       .csv("/Volumes/dbacademy/default/tutorials/maintenance_logs.csv")
                    ) 
                        

In [0]:
display(df_telemetry_json)
display(df_maintenance_logs)

bike_model,date,ride_id,telemetry,warnings
CB300R-SpartanRed,2026-04-26,R-101,"List(OFF, 0, 85)","List(PARKED_TIP_OVER, COSMETIC_DAMAGE)"
CB300R-SpartanRed,2026-04-27,R-102,"List(ON, 45, 15)",List()
CBR650R,2026-04-26,R-103,"List(ON, 120, 40)","List(HIGH_RPM, ABS_ENGAGED)"
CB300R-SpartanRed,2026-05-02,R-104,"List(ON, 82, 22)",List()


ride_id,service_action,cost_inr
R-101,Replace structural fairing and lever,4500
R-102,None,0
R-103,Chain lube and tension,500
R-104,General checkup,0


In [0]:
df_filt_telemetry = (df_telemetry_json
                    .filter(F.col("bike_model") == "CB300R-SpartanRed")
                    .withColumn("Incident_Tilt", F.col("telemetry.tilt_angle"))
                    .filter(F.col("Incident_Tilt") > 60)
                    .withColumn("warnings", F.explode(F.col("warnings")))
                ).join(df_maintenance_logs, on='ride_id', how ='inner')
                
display(df_filt_telemetry.select("date", "Incident_Tilt", "warnings", "service_action", "cost_inr"))

date,Incident_Tilt,warnings,service_action,cost_inr
2026-04-26,85,PARKED_TIP_OVER,Replace structural fairing and lever,4500
2026-04-26,85,COSMETIC_DAMAGE,Replace structural fairing and lever,4500
